# Get TYPED only variants 

**Version**: 1.0.0  
**Last iteration**: 01-DEC-2025  

## Imports


In [ ]:
import os
from datetime import date
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

In [2]:
d = date.today()

print(f'''
Last iteration: {d}

pandas=={pd.__version__}
numpy=={np.__version__}
''')


Last iteration: 2025-12-02

pandas==2.3.2
numpy==2.1.3



## Set directories and variables

### Common paths

In [ ]:
# Directories

# Hestia NGS Software
tools = "/path/to/tools"

# Main directory
main_dir = "/path/to/analysis"
MAIN_DIR = main_dir     ### alias

# Data directory
data_dir = f"{main_dir}/output/data"
DATA_DIR = data_dir     ### alias

# Raw data directory
raw_dir = f"{data_dir}/RAW"
RAW_DIR = raw_dir     ### alias

# Imputed data directory
impt_dir = f"{data_dir}/IMPUTED"
IMPT_DIR = raw_dir     ### alias

# Imputed/softcalls data directory
soft_dir = f"{data_dir}/IMPUTED/softcalls"
SOFT_DIR = raw_dir     ### alias

# Typed onlye data directory
typed_dir = f"{data_dir}/IMPUTED/typed"
TYPED_DIR = raw_dir     ### alias

# Meta data (covariate, population, ancestry labels, etc.)
meta_dir = f"{data_dir}/META"
META_DIR = meta_dir     ### alias

### Paths to software and tools

In [ ]:
# Plink1.9 and Plink2.0 path
plink1 = f"{tools}/plink_linux_x86_64_20250615/plink"
plink2 = f"{tools}/plink2_linux_avx2_20250609/plink2"

# KING path
king = f"{tools}/king/king"

# Eagle
eagle = f"{tools}/eagle/Eagle_v2.4.1/eagle"

# Bcftools
bcftools = f"{tools}/bcftools-1.19/bcftools"
# Export bcftools plugins as env variable
os.environ["BCFTOOLS_PLUGINS"] = f"{tools}/bcftools-1.19/plugins"

# BGZIP
bgzip = f"{tools}/tabix-0.2.6/bgzip"

# GWASQC path
gwasqc = f"{tools}/GWASQC/main.py"

# NAToRA path
natora = f"{tools}/NAToRA_Public/NAToRA_Public.py"

### Input, output, covariate files

In [ ]:
# Input file path without suffix 
rawFile = f"{raw_dir}/CATPD.update_pheno"
inputPfile = f"{raw_dir}/CATPD"      ### Updated IDs, Sex and added PD Pheno

# Covariate file
covar = f"{meta_dir}/CATPD.cov"

# Populations/Ancestries file
pops = f"{meta_dir}/CATPD.pop"

# Pheno Name
pheno = "STATUS"

# Keep file
keep = f"{meta_dir}/CATPD.keep"

# Chromosomes as list
chromosomes = list(range(1,23))

# Threads by default
threads = 1

# Output directory path and output prefix
outdir = f"{raw_dir}/GWASQC"
os.makedirs(outdir, exist_ok=True)
output = "CATPD"

In [ ]:
# FASTA
# Hestia NGS_Reference
reference_path = f"/path/to/reference/"

# Hg38
fasta = f"{reference_path}/fasta/hg38/Homo_sapiens_assembly38.fasta"

# gnomAD files
gnomad = f"{reference_path}/gnomAD/gnomadOnlyAF_onePercent_*.vcf.gz"

## Download imputed .zip files after completion

Set the current downloaded area as a working directory.  
Download imputed .zip files using wither wget or curl.

In [ ]:
os.makedirs(f"{impt_dir}/zip", exist_ok=True)
os.chdir(f"{impt_dir}/zip")

### Unzip files with password

<div class="alert alert-block alert-info">
⚠️ Best way to run this is use bash command with nohup for all chr in parallel
</div>

In [ ]:
%%bash

cd ${impt_dir}/zip
soft_dir=${impt_dir}/softcalls

for c in {1..22}
do
    echo -e "nohup unzip -P ${password} chr_${c}.zip -d {soft_dir} > chr{c}.log 2>&1 &"
done

## Get softcalls and typed-only variants from imputed data

Impute dosage data. Use bcftools to extract only typed variants with ```bcftools view -i \"INFO/TYPED=1\"``` and then set all variants as chr:pos with ```bcftools annotate --set-id \"%CHROM:%POS\"```. Then concatenate all chromosomal files with ```bcftools concat -f file_list -Oz -o  ALLCHR_OnlyTyped.vcf.gz``` where file_list is just the ```ls -1``` for all vcf.gz.   
After that make the vcf.gz into plink1.9 and plink2.0 filetypes. 
Use bcftools or plink2 to extract softcalls from imputed data, set all variants same as in TYPED.

Extract typed only variants per each chromosome.

In [ ]:
%%bash

typed_dir=${impt_dir}/typed
for c in {1..22}
do 
    echo -e "bcftools view -i 'INFO/TYPED=1' --threads 1  -Oz -o $typed_dir/chr${c}.dose.typed.vcf.gz & "
done

Create a file list with all typed only variant files per chromosome.

In [ ]:
%%bash

typed_dir=${impt_dir}/typed
for c in {1..22}
do 
    echo -e "chr${c}.dose.typed.vcf.gz" >> file_list
done

Concatenate all vcf.g files into one.

In [ ]:
%%bash

typed_dir=${impt_dir}/typed
bcftools concat -f file_list -Oz -o  $typed_dir/chr1_22.typed.vcf.gz